# Lentils × Dinomaly: training tutorial (RGB front-end)

A lentil conveyor should carry nothing but lentils. In practice it also carries stones,
aluminium shards, paper snippets, rubber, and the occasional fly. Cataloguing every possible
foreign object is hopeless, which is what makes this an **anomaly detection** problem: learn
what normal lentils look like, and flag whatever deviates.

Dinomaly does exactly that. It is reconstruction-based and fully unsupervised: a frozen DINOv2
vision transformer extracts features, and a small decoder learns to reconstruct those features
**on normal frames only**. At inference, regions the decoder cannot reconstruct are the
anomalies. No foreign object is ever seen during training.

This notebook trains Dinomaly on the 61-band VNIR lentils dataset with an **RGB front-end**:
a fixed-wavelength selector collapses the cube to 3 channels at 650/550/450 nm, so the ViT
runs in its native image domain.

**Pipeline at a glance:**

```
AnomalyDataNode ──► MinMaxNormalizer ──► FixedWavelengthSelector(650/550/450 nm)
     ──► DinomalyDetector ──► {QuantileBinaryDecider, AnomalyDetectionMetrics,
         AnomalyAUROCMetrics} ──► TensorBoardMonitorNode
```

Sibling notebooks: `lentils_cir_train_tutorial.ipynb` (CIR front-end),
`lentils_adaclip_bands_train_tutorial.ipynb` (AdaCLIP-selected bands), and
`lentils_inference_tutorial.ipynb` (evaluation with per-class AUROC).

> **Prerequisites**
>
> 1. Install cuvis-ai-dinomaly with the examples extra: `uv sync --extra examples`.
> 2. From the repo root, launch the notebook with `uv run jupyter lab`.

In [ ]:
# Colab bootstrap: no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q cuvis-ai cuvis-ai-dinomaly "cuvis-ai-dataloader[cu3s,coco]"

    import torch

    if not torch.cuda.is_available():
        print(
            "WARNING: No GPU detected. Switch via Runtime > Change runtime type > T4 GPU. "
            "Dinomaly training on CPU is very slow."
        )

In [ ]:
# ruff: noqa: E402
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Image, Video, display
from loguru import logger

from cuvis_ai.node.deciders.binary_decider import QuantileBinaryDecider
from cuvis_ai.node.channel_selector import FixedWavelengthSelector
from cuvis_ai.node.data import AnomalyDataNode
from cuvis_ai.node.metrics import AnomalyDetectionMetrics
from cuvis_ai.node.monitor import TensorBoardMonitorNodesam
from cuvis_ai.node.normalization import MinMaxNormalizer
from cuvis_ai.node.video import ToVideoNode
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.data.splits_io import load_splits
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils import restore_trainrun
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dataloader.data.npz_converter import convert_split_manifest
from cuvis_ai_dinomaly.node.auroc_metrics import AnomalyAUROCMetrics
from cuvis_ai_dinomaly.node.dinomaly_detector import DinomalyDetector
from cuvis_ai_dinomaly.node.dinomaly_train_loss_bridge import DinomalyTrainLossBridge
from cuvis_ai_schemas.pipeline import PipelineMetadata
from cuvis_ai_schemas.training import (
    CallbacksConfig,
    DataConfig,
    DataSplitConfig,
    ModelCheckpointConfig,
    OptimizerConfig,
    TrainingConfig,
    TrainRunConfig,
)

In [ ]:
device = torch.device("cpu")
# Pick the best available torch device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## 1 · Fetch + prepare the dataset

The XMR Industrial Foreign Object Detection (Lentils) dataset (~57 GB) lives on Hugging Face
Hub: 15 merged cu3s sessions recorded on three acquisition days, 61 spectral bands (430 to
910 nm) per pixel, and pixel-level COCO masks for 7 foreign-object classes.

Dinomaly trains on normals only, so the dataset ships a dedicated split manifest
(`splits_dinomaly.csv`):

| split | frames | anomalous | role |
|---|---|---|---|
| train | 308 | 0 | Dinomaly training |
| val | 148 | 84 | calibration / threshold |
| test | 180 | 112 | evaluation |
| adaclip_train | 500 | 500 | held out for the supervised baseline |

Two steps, each skipped when its output already exists:

1. **Fetch** the raw dataset with `PublicDatasets.download_dataset` (skips when the folder is
   already on disk).
2. **Convert** the manifest's frames to per-frame NPZ, via `convert_split_manifest` from
   cuvis-ai-dataloader. It writes two artifacts: a `universe.csv` (the sample universe, listing which `.npz` holds
   which `(source, frame)`) and a `splits.json` assignment. The `splits.json` is core's
   `DataSplitConfig`, the same selector-based split file the CuvisNEXT split designer loads and
   saves, so one file drives training here and in the GUI. Re-running is cheap: already-converted
   frames are reused, so this regenerates the two artifacts in seconds.

In [ ]:
NPZ_DIR = Path("outputs/npz_local")
SPLITS_JSON = NPZ_DIR / "splits.json"
UNIVERSE_CSV = NPZ_DIR / "universe.csv"

dataset_dir = Path("/content/data") if IN_COLAB else Path("../../data")
raw_dir = dataset_dir / "XMR_Industrial_Foreign_Object_Detection_Lentils"

if SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file():
    print(f"Split artifacts already present: {SPLITS_JSON}")
else:
    _ = PublicDatasets.download_dataset(
        "industrial_fod_lentils",
        download_path=str(dataset_dir),
        force=False,
    )

In [ ]:
if not (SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file()):
    result = convert_split_manifest(
        raw_dir / "splits_dinomaly.csv",
        raw_dir,
        NPZ_DIR,
        universe_csv=UNIVERSE_CSV,
        splits_json=SPLITS_JSON,
    )
    SPLITS_JSON, UNIVERSE_CSV = result.splits_json, result.universe_csv
print("Splits JSON:", SPLITS_JSON)
print("Index CSV:  ", UNIVERSE_CSV)

## 2 · How Dinomaly works

Dinomaly is reconstruction-based anomaly detection on ViT features:

- A **frozen DINOv2 ViT-B/14 encoder** turns each frame into patch-token features. It is
  never trained here; its weights download automatically on first use.
- A small **bottleneck MLP** and an 8-block **transformer decoder** are the trainable parts.
  They learn to reconstruct the encoder's features on normal lentils.
- The **anomaly map** is the cosine distance between encoder and decoder features: regions the
  decoder cannot reconstruct (it never saw them in training) score high. Out come a pixel map
  (`scores`, one value per pixel) and a per-frame `anomaly_score`.

Encoder, bottleneck, and decoder live inside the single `DinomalyDetector` node, following the
wrapper-first convention for pretrained networks: they are hyperparameters of one node, not
separate nodes. What you mix and match at pipeline level sits around it: the selector
front-end (this notebook uses fixed RGB wavelengths; the siblings use CIR and AdaCLIP-selected
bands), the decider, and the metric nodes.

## 3 · Tutorial configuration

Edit these variables to customise the run.

- **`MAX_EPOCHS`**: 1 gives a quick smoke run that exercises the whole path end to end. Set
  20 to 50 for a real model.
- **`IMAGE_SIZE`**: square side the detector works at; a multiple of the ViT patch size 14.
- **`output_dir`**: everything this notebook produces lands here (pipeline yaml, trainrun
  yaml, checkpoints, TensorBoard logs, preview video, trained weights).

In [ ]:
MAX_EPOCHS = 1  # 20-50 for a real run
IMAGE_SIZE = 448  # multiple of 14

output_dir = Path("outputs/lentils_rgb_run")
output_dir.mkdir(parents=True, exist_ok=True)

pipeline_yaml_path = output_dir / "dinomaly_lentils_rgb.yaml"
trainrun_yaml_path = output_dir / "trainrun.yaml"
preview_video_path = output_dir / "test_preview.mp4"

print(f"Epochs:      {MAX_EPOCHS}")
print(f"Image size:  {IMAGE_SIZE}")
print(f"Splits JSON: {SPLITS_JSON}")
print(f"Index CSV:   {UNIVERSE_CSV}")
print(f"Output dir:  {output_dir}")

## 4 · Preview the data, as a pipeline

Before training, look at what the model will see. Even the preview is a Cuvis.AI pipeline:
the same `AnomalyDataNode` and `FixedWavelengthSelector` the training pipeline uses, plus a
`ToVideoNode` sink. Computation happens in nodes; matplotlib only renders node outputs.

```
AnomalyDataNode ──► FixedWavelengthSelector ──► ToVideoNode
```

Three cells: build the pipeline and show its graph, sanity-check one labeled frame, then
render the whole test split to a video. (The sanity check writes a short MP4 as a side
effect; the full render afterwards overwrites it.)

In [ ]:
preview_pipeline = CuvisPipeline("lentils_rgb_preview")

preview_data = AnomalyDataNode(normal_class_ids=[0], name="anomaly_data")
preview_selector = FixedWavelengthSelector(
    target_wavelengths=(650.0, 550.0, 450.0), name="rgb_selector"
)
preview_video = ToVideoNode(
    output_video_path=str(preview_video_path), frame_rate=5.0, name="to_video"
)

preview_pipeline.connect(
    (preview_data.outputs.cube, preview_selector.cube),
    (preview_data.outputs.wavelengths, preview_selector.wavelengths),
    (preview_selector.rgb_image, preview_video.rgb_image),
)

graph_png = preview_pipeline.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{preview_pipeline.name}.png"),
)
display(Image(str(graph_png)))

In [ ]:
preview_datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
)

preview_pipeline.to(device)
preview_outputs = Predictor(pipeline=preview_pipeline, datamodule=preview_datamodule).predict(
    max_batches=12, collect_outputs=True
)


def _gt_mask(out):
    mask = out.get(("anomaly_data", "mask"))
    return None if mask is None else mask[0, ..., 0].cpu().numpy()


picked = next(
    (o for o in preview_outputs if _gt_mask(o) is not None and _gt_mask(o).any()),
    preview_outputs[0],
)
rgb = picked[("rgb_selector", "rgb_image")][0].cpu().numpy()
mask = _gt_mask(picked)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(rgb)
ax[0].set_title("RGB view (650/550/450 nm)")
ax[0].axis("off")
ax[1].imshow(rgb)
if mask is not None and mask.any():
    ax[1].contour(mask, levels=[0.5], colors="red", linewidths=1.2)
ax[1].set_title("scene + GT contour")
ax[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
video_datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
)

preview_pipeline.to(device)
Predictor(pipeline=preview_pipeline, datamodule=video_datamodule).predict(collect_outputs=False)

if not preview_video_path.exists():
    raise RuntimeError(f"Preview video was not created: {preview_video_path}")
display(Video(str(preview_video_path), embed=IN_COLAB, width=640))

## 5 · Build the training pipeline

Nine nodes:

- `AnomalyDataNode` turns each batch into a float32 cube plus a wavelength vector, and maps
  the multi-class GT mask to a binary anomaly mask (classes in `normal_class_ids` become 0,
  everything else 1).
- `MinMaxNormalizer` scales the cube per channel; its running bounds are seeded by the
  statistical training phase.
- `FixedWavelengthSelector` picks the bands nearest 650/550/450 nm: a 3-channel image.
- `DinomalyDetector` wraps the frozen DINOv2 encoder plus the trainable bottleneck and
  decoder; it emits the pixel `scores` map, the per-frame `anomaly_score`, and a
  `training_loss`.
- `DinomalyTrainLossBridge` exposes that loss to the trainer.
- `QuantileBinaryDecider` binarizes the score map at the 99.5th percentile.
- `AnomalyDetectionMetrics` (IoU/Dice on the binarized map) drives checkpointing, and
  `AnomalyAUROCMetrics` streams pixel and image AUROC each epoch; both read the GT mask.
- `TensorBoardMonitorNode` logs the metrics.

The pipeline is saved to yaml right away: the trainrun in the next section references it by
path.

In [ ]:
pipeline = CuvisPipeline("dinomaly_lentils_rgb")

data_node = AnomalyDataNode(normal_class_ids=[0], name="anomaly_data")
normalizer = MinMaxNormalizer(eps=1e-6, use_running_stats=True, max_initialization_frames=20)
selector = FixedWavelengthSelector(target_wavelengths=(650.0, 550.0, 450.0), name="rgb_selector")
dinomaly = DinomalyDetector(
    encoder_name="dinov2reg_vit_base_14",
    bottleneck_dropout=0.2,
    decoder_depth=8,
    image_size=IMAGE_SIZE,
    crop_size=IMAGE_SIZE,
    use_center_crop=False,
    input_channels=3,
    name="dinomaly_detector",
)
loss_bridge = DinomalyTrainLossBridge(weight=1.0, name="dinomaly_train_loss")
decider = QuantileBinaryDecider(quantile=0.995, name="decider")
metrics_node = AnomalyDetectionMetrics(name="metrics_anomaly")
auroc_node = AnomalyAUROCMetrics(name="metrics_auroc")
tb = TensorBoardMonitorNode(output_dir=str(output_dir / "tensorboard"), run_name=pipeline.name)

pipeline.connect(
    (data_node.outputs.cube, normalizer.data),
    (normalizer.normalized, selector.cube),
    (data_node.outputs.wavelengths, selector.wavelengths),
    (selector.rgb_image, dinomaly.rgb_image),
    (dinomaly.outputs.training_loss, loss_bridge.raw_loss),
    (dinomaly.outputs.scores, decider.logits),
    (dinomaly.outputs.scores, metrics_node.logits),
    (decider.decisions, metrics_node.decisions),
    (data_node.outputs.mask, metrics_node.targets),
    (metrics_node.metrics, tb.metrics),
    (dinomaly.outputs.scores, auroc_node.scores),
    (data_node.outputs.mask, auroc_node.targets),
    (dinomaly.outputs.anomaly_score, auroc_node.anomaly_score),
)

pipeline.save_to_file(
    str(pipeline_yaml_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description="Dinomaly on lentils VNIR NPZ, RGB fixed-wavelength selector (650/550/450 nm).",
        tags=["dinomaly", "anomalib", "lentils", "rgb", "hyperspectral"],
        author="cuvis.ai",
    ),
)
print("Pipeline saved:", pipeline_yaml_path)

In [ ]:
graph_png = pipeline.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{pipeline.name}.png"),
)
display(Image(str(graph_png)))

## 6 · Train, via a trainrun

Training is driven by Cuvis.AI's **trainrun** feature rather than hand-assembled trainer
objects. A `TrainRunConfig` bundles everything a run needs: the pipeline (referenced by the
yaml path saved above), the data module, the training schedule, and which nodes provide the
loss and the metrics. `restore_trainrun` then executes the full sequence: statistical
initialization (the MinMax normalizer bounds), gradient training (bottleneck and decoder
train; the DINOv2 encoder stays frozen), saving the trained pipeline, and a validation plus
test pass to populate TensorBoard.

The saved yaml reproduces the run from a terminal:

```bash
uv run restore-trainrun --trainrun-path outputs/lentils_rgb_run/trainrun.yaml --mode train
```

In [ ]:
trainrun = TrainRunConfig(
    name="dinomaly_lentils_rgb",
    pipeline=pipeline_yaml_path.name,  # resolved relative to the trainrun yaml
    data=DataConfig(
        data_module="npz_multi",
        batch_size=1,
        num_workers=0,
        # splits_path points at the splits.json; core loads it (absolute, so it resolves
        # regardless of where the trainrun runs from). universe is the universe lookup.
        splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
        params={"universe": str(UNIVERSE_CSV.resolve())},
    ),
    training=TrainingConfig(
        seed=42,
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        devices=1,
        default_root_dir=str(output_dir),
        precision="32-true",
        enable_progress_bar=True,
        enable_checkpointing=True,
        log_every_n_steps=10,
        check_val_every_n_epoch=1,
        gradient_clip_val=0.1,
        optimizer=OptimizerConfig(name="adamw", lr=2e-3, weight_decay=1e-4, betas=[0.9, 0.999]),
        callbacks=CallbacksConfig(
            checkpoint=ModelCheckpointConfig(
                dirpath=str(output_dir / "checkpoints"),
                monitor="metrics_anomaly/iou",
                mode="max",
                save_top_k=1,
                save_last=True,
                filename="{epoch:02d}",
            )
        ),
    ),
    loss_nodes=["dinomaly_train_loss"],
    metric_nodes=["metrics_anomaly", "metrics_auroc"],
    unfreeze_nodes=["dinomaly_detector"],
    output_dir=str(output_dir),
)
trainrun.save_to_file(trainrun_yaml_path)
print("Trainrun saved:", trainrun_yaml_path)

restore_trainrun(trainrun_yaml_path, mode="train")

## 7 · What the run produced

`restore_trainrun` saved the trained pipeline itself (yaml plus weights under
`trained_models/`); there is nothing to persist by hand.

In [ ]:
trained_dir = output_dir / "trained_models"
for artifact in sorted(trained_dir.glob("*")):
    print(f"{artifact}  ({artifact.stat().st_size / 1e6:.0f} MB)")

trained_yaml = trained_dir / f"{pipeline.name}_restored.yaml"
assert trained_yaml.is_file(), f"expected trained pipeline at {trained_yaml}"

## 8 · Next

Evaluate the trained pipeline on the 180-frame test split with
**`lentils_inference_tutorial.ipynb`**, pointing its pipeline directory at
`outputs/lentils_rgb_run/trained_models`. A 1-epoch smoke model will score modestly; for a
real model set `MAX_EPOCHS = 20` (or 50) in section 3 and rerun from section 6, either here or
with the `restore-trainrun` command above. TensorBoard logs live under
`outputs/lentils_rgb_run/tensorboard`.